---
title: 'Phase 1 validation: testing a basic model'
jupyter:
  jupytext:
    text_representation:
      extension: .qmd
      format_name: quarto
      format_version: '1.0'
      jupytext_version: 1.19.5
  kernelspec:
    display_name: Python 3 (ipykernel)
    language: python
    name: python3
---


The goal of this notebook is to show that a classical-quantum model generative model can actually learn, using a simple model. For this, we would be using the torch library to create the classical part of our network, and add the quantum part through the functions and objects we created in `data.py` and `models.py`. You can find a more detailed explanation in the `docs`folder, including a schema of our model. Here, we will focus on explaining what we are doing instead of why.


In [1]:
# Add the src folder to the path, regardless of where the kernel's cwd is
import sys, pathlib

for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (parent / "src" / "data.py").exists():
        sys.path.insert(0, str(parent / "src"))
        break
else:
    raise RuntimeError("Could not locate the src/ folder")


In [2]:
from torch.nn import ReLU, Tanh

from data import * 
from models import *

import torch
import numpy as np

# We build the model
model = nn.Sequential(
    nn.Linear(6, 6),
    nn.Tanh(),
    QuantumCircuit(),
    nn.ReLU(),
    nn.Linear(56, 2),
)


Now, before training the model, we have to generate the desired output so we can train using the MDD loss.


In [3]:
desired_output = two_gaussian(256)
print(desired_output[:5])


tensor([[-1.3378, -0.3457],
        [-1.0752, -0.1302],
        [-0.7454,  0.2076],
        [-1.0948, -0.6346],
        [-0.9033, -0.3790]])


Let's also view this in a graph to ensure that this is working.


In [4]:
import matplotlib.pyplot as plt

points = desired_output.numpy() # tensor -> numpy array
plt.scatter(points[:, 0], points[:, 1])
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()


As we can see, we have two gaussians centered around (-1, 0) and (1, 0).

Now let's test the random_input_numbers function.


In [5]:
random_numers_test = []
for i in range(256): # Here, we make a loop instead of just calling random_input_numbers once, because of the seed. 
    # It is best to test it like a real simulation
    random_num_test = random_input_numbers(1, seed=i) # Also, we set the seed as i because taking the same seed
    # would just output 256 times the same numbers.
    random_numers_test.append(random_num_test.numpy())

all_values = np.array(random_numers_test).flatten()
plt.hist(all_values, bins=30)
plt.xlabel("value")
plt.ylabel("count")
plt.show()


As we can see here, we do have a random normal distribution, so the random_input_numbers function is properly working.

Now, before doing any training, let's see what result we get from running our model once.


In [6]:
outputs_test = []
for i in range(256):
    x = random_input_numbers(1, seed=i)
    y = model(x)
    outputs_test.append(y)

print(outputs_test[:5])


[tensor([[-0.1241, -0.0877]], grad_fn=<AddmmBackward0>), tensor([[-0.1401, -0.0743]], grad_fn=<AddmmBackward0>), tensor([[-0.1337, -0.0769]], grad_fn=<AddmmBackward0>), tensor([[-0.1249, -0.0777]], grad_fn=<AddmmBackward0>), tensor([[-0.1292, -0.0816]], grad_fn=<AddmmBackward0>)]


In [7]:
xs = [t[0, 0].item() for t in outputs_test]
ys = [t[0, 1].item() for t in outputs_test]

plt.figure(figsize=(6, 6))
plt.scatter(xs, ys, alpha=0.6, s=15)
plt.xlabel("Output dim 0")
plt.ylabel("Output dim 1")
plt.title("Model outputs for 256 random inputs")
plt.grid(True, alpha=0.3)
plt.show()


As we can see, this is what our untrained model outputs. It looks nothing like the two gaussians we want. 


Let's just optimize a little bit computation time before that, by making batches of 256 at once. Now we have one tensor, and it's easier to manipulate it.


In [8]:
x_opti = random_input_numbers(256, seed=42)
y_opti = model(x_opti)

print(y_opti[:5])


tensor([[-0.1408, -0.0691],
        [-0.1263, -0.0852],
        [-0.1252, -0.0879],
        [-0.1409, -0.0703],
        [-0.1515, -0.0581]], grad_fn=<SliceBackward0>)


Now is the time to test the loss function.


In [9]:
from losses import mmd_loss

loss = mmd_loss(y_opti, desired_output)
print(loss)


tensor(0.3642, grad_fn=<SubBackward0>)


In [10]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


for i in range(700):
    input = random_input_numbers(256, seed=i)
    des_output = two_gaussian(256, seed=42)

    optimizer.zero_grad()
    y_train = model(input)
    loss_training = mmd_loss(y_train, des_output)
    loss_training.backward()
    optimizer.step()

    print(f"step {i}, loss = {loss_training.item():.4f}")


step 0, loss = 0.3663
step 1, loss = 0.3659
step 2, loss = 0.3655
step 3, loss = 0.3654
step 4, loss = 0.3652
step 5, loss = 0.3647


step 6, loss = 0.3644
step 7, loss = 0.3640
step 8, loss = 0.3638
step 9, loss = 0.3636
step 10, loss = 0.3634
step 11, loss = 0.3632
step 12, loss = 0.3629
step 13, loss = 0.3625
step 14, loss = 0.3624


step 15, loss = 0.3622
step 16, loss = 0.3619
step 17, loss = 0.3617
step 18, loss = 0.3614
step 19, loss = 0.3613
step 20, loss = 0.3612
step 21, loss = 0.3610
step 22, loss = 0.3608


step 23, loss = 0.3607
step 24, loss = 0.3605
step 25, loss = 0.3604
step 26, loss = 0.3601
step 27, loss = 0.3600
step 28, loss = 0.3599
step 29, loss = 0.3598
step 30, loss = 0.3597


step 31, loss = 0.3595
step 32, loss = 0.3595
step 33, loss = 0.3594
step 34, loss = 0.3593
step 35, loss = 0.3592
step 36, loss = 0.3591


...

step 669, loss = 0.0058
step 670, loss = 0.0093
step 671, loss = 0.0066
step 672, loss = 0.0064
step 673, loss = 0.0071
step 674, loss = 0.0081
step 675, loss = 0.0043
step 676, loss = 0.0070
step 677, loss = 0.0035


step 678, loss = 0.0059
step 679, loss = 0.0061
step 680, loss = 0.0084
step 681, loss = 0.0049
step 682, loss = 0.0070
step 683, loss = 0.0033
step 684, loss = 0.0062


step 685, loss = 0.0070
step 686, loss = 0.0049
step 687, loss = 0.0044
step 688, loss = 0.0089
step 689, loss = 0.0035
step 690, loss = 0.0074


step 691, loss = 0.0045
step 692, loss = 0.0069
step 693, loss = 0.0082
step 694, loss = 0.0132
step 695, loss = 0.0047
step 696, loss = 0.0048
step 697, loss = 0.0052
step 698, loss = 0.0072


step 699, loss = 0.0046


In [11]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)


0.weight torch.Size([6, 6])
0.bias torch.Size([6])
2.quantum_layer.el torch.Size([60])
4.weight torch.Size([2, 56])
4.bias torch.Size([2])


In [12]:
total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_trainable:,}")


Trainable parameters: 216


In [14]:
input_test = random_input_numbers(256, 43)
y_test = model(input_test)

xs_test = y_test[:,0]
ys_test = y_test[:,1]

xs_input = input_test[:,0]
ys_input = input_test[:,1]

plt.figure(figsize=(6, 6))
plt.scatter(xs_test.detach().numpy(), ys_test.detach().numpy(), alpha=0.5)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Sortie du modèle")
plt.show()


plt.figure(figsize=(6, 6))
plt.scatter(xs_input.detach().numpy(), ys_input.detach().numpy(), alpha=0.5)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Input")
plt.show()
